In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table, join
from astropy.io import fits
from sdss_access import Access

# Read the DAP file (Helix Nebula DR20 Release)

In [2]:
access = Access(release='dr20', verbose=True)
drpver = "1.2.0"
dapver = '1.2.0.251218' 
kind   = 'dap' 
tileid = '11111'
mjd    = 60191
expnum = '00004297'
i_exp  = int(expnum)
rsp    = 'rsp108'
sn     = 'sn20'

path_to_data_file = access.full('lvm_dap',  drpver = drpver, 
                                dapver = dapver, kind = kind, 
                                tileid = tileid, mjd = mjd, 
                                expnum = i_exp, rspid = rsp, 
                                snlevel = sn, daptype = 'dap')
print('Looking for file in',path_to_data_file)

if os.path.exists(path_to_data_file):
    fits.info(path_to_data_file)
else:
    access.remote()
    kind = 'dap'    
    access.add('lvm_dap', drpver = drpver, dapver = dapver, kind = kind, tileid = tileid, mjd = mjd, 
               expnum = i_exp, rspid = rsp, snlevel = sn, daptype = 'dap')
    access.set_stream()
    access.commit()
    if not os.path.isfile(path_to_data_file):
        print('DAP output file could not be retrieved!')
dap_hdu=fits.open(path_to_data_file)

Looking for file in /Users/tonywong/Data/sdss_lvm/sdsswork/dr20/spectro/lvm/analysis/1.2.0/1.2.0.251218/0011XX/11111/60191/00004297/dap-rsp108-sn20-00004297.dap.fits
Filename: /Users/tonywong/Data/sdss_lvm/sdsswork/dr20/spectro/lvm/analysis/1.2.0/1.2.0.251218/0011XX/11111/60191/00004297/dap-rsp108-sn20-00004297.dap.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     931   ()      
  1  PT            1 BinTableHDU     21   1754R x 6C   [9A, D, D, L, K, K]   
  2  RSP           1 BinTableHDU    101   1754R x 31C   [9A, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D]   
  3  COEFFS        1 BinTableHDU     35   189432R x 13C   [9A, K, K, D, D, D, D, D, D, D, D, D, D]   
  4  PM_ELINES     1 BinTableHDU     29   26310R x 10C   [9A, K, 10A, D, D, D, D, D, D, D]   
  5  NP_ELINES_B    1 BinTableHDU   1835   1754R x 913C   [9A, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D,

# Extract the tables, first nine have per-fiber data

In [3]:
tab_PT     = Table(dap_hdu['PT'].data)
tab_RSP    = Table(dap_hdu['RSP'].data)
tab_COEFFS = Table(dap_hdu['COEFFS'].data)
tab_PE     = Table(dap_hdu['PM_ELINES'].data)
tab_NPE_B  = Table(dap_hdu['NP_ELINES_B'].data)
tab_NPE_R  = Table(dap_hdu['NP_ELINES_R'].data)
tab_NPE_I  = Table(dap_hdu['NP_ELINES_I'].data)
tab_KEL    = Table(dap_hdu['PM_KEL'].data)
tab_SIGMA  = Table(dap_hdu['ELINES_SIGMA_CHI'].data)
tab_CHIAVG = Table(dap_hdu['ELINES_CHI2_AVG'].data)
tab_INFO   = Table(dap_hdu['INFO'].data)

# Sky positions table

In [4]:
print('\nPT table:\n',tab_PT)


PT table:
     id            ra                 dec         mask fiberid exposure
--------- ------------------ ------------------- ---- ------- --------
   4297.3  337.4169474614286 -21.084344018449997 True       3     4297
   4297.4 337.42638153117906  -21.07898431939437 True       4     4297
   4297.5 337.43581492731073 -21.073624095271583 True       5     4297
   4297.6  337.4452142099591 -21.068264089182854 True       6     4297
   4297.7 337.45464625878174 -21.062902818356775 True       7     4297
   4297.8 337.46407763267024  -21.05754102404877 True       8     4297
   4297.9 337.47350833118605   -21.0521787067879 True       9     4297
  4297.10 337.48293835389075   -21.0468158671032 True      10     4297
  4297.11 337.49233426716734  -21.04145325730424 True      11     4297
  4297.12    337.50176293819  -21.03608937619941 True      12     4297
      ...                ...                 ...  ...     ...      ...
4297.1934  337.2728561387537 -20.665038371654944 True    1934    

# RSP table

In [5]:
print('\nRSP table:\n',tab_RSP)
renames = {
           'Av'         : 'Av_st',
           'e_Av'       : 'e_Av_st',
           'z'          : 'z_st',
           'e_z'        : 'e_z_st',
           'disp'       : 'disp_st',
           'e_disp'     : 'e_disp_st',
           'flux'       : 'flux_st',
           'redshift'   : 'redshift_st',
           'med_flux'   : 'med_flux_st',
           'e_med_flux' : 'e_med_flux_st',
           'sys_vel'    : 'vel_st'
}
for key in renames.keys():
    tab_RSP.rename_column(key, renames[key])
print('\nRSP table columns:\n',tab_RSP.colnames)


RSP table:
     id         MIN_CHISQ      ...       log_ML            log_Mass     
--------- ------------------- ... ------------------ ------------------
   4297.3   281.1142545729355 ...  10.84083474091801 19.040630766008036
   4297.4  264.60330556432086 ... 11.013698503708444  18.83911321277269
   4297.5  0.5538120363764494 ... 11.013698503708444 18.839145557883292
   4297.6  0.5707569727456165 ... 10.679959207853686 18.432586670797342
   4297.7 0.45241537722213293 ...  9.558578597984615  17.69734827787621
   4297.8 0.48184623472867033 ...  9.470867281824013   17.4684574190109
   4297.9 0.45486763990437057 ...  9.433725541390281  17.48195935885809
  4297.10 0.41398424379025417 ...  9.831825849712523 17.985101689237695
  4297.11  0.4736389129144122 ...  9.492424858639431    17.668331731808
  4297.12 0.46172214253808447 ...  9.558578597984615 17.614278851151333
      ...                 ... ...                ...                ...
4297.1934  0.5458958768235249 ...  8.62228128299282

# COEFFS table; create separate columns for each of the 108 RSPs

In [6]:
print('\nCOEFFS table:\n',tab_COEFFS)
tab_COEFFS.remove_columns(['id_fib'])
renames = {
           'rsp'        : 'id_rsp',
           'TEFF'       : 'Teff_rsp',
           'LOGG'       : 'Log_g_rsp',
           'META'       : 'Fe_rsp',
           'ALPHAM'     : 'alpha_rsp',
           'COEFF'      : 'W_rsp',
           'Min.Coeff'  : 'min_W_rsp',
           'log(M/L)'   : 'log_ML_rsp',
           'AV'         : 'Av_rsp',
           'N.Coeff'    : 'n_W_rsp',
           'Err.Coeff'  : 'e_W_rsp'
}
for key in renames.keys():
    tab_COEFFS.rename_column(key, renames[key])
print('\nCOEFFS renamed table columns:\n',tab_COEFFS.colnames)
df0 = tab_COEFFS.to_pandas()
df0.set_index(['id','id_rsp'], inplace=True)
df1 = df0.unstack(level=-1)
df1.columns = [f"{c[0]}_{c[1]}" for c in df1.columns]
df2 = df1.reset_index()
tab_C_ord = Table.from_pandas(df2)
print('\nC_ord table:\n',tab_C_ord)
print('\nC_ord table columns:\n',tab_C_ord.colnames)


COEFFS table:
     id    id_fib rsp  TEFF   LOGG  ... Min.Coeff log(M/L)  AV  N.Coeff Err.Coeff
--------- ------ --- ------ ------ ... --------- -------- ---- ------- ---------
   4297.3      0   0 3.8065 4.0113 ...       0.0  14.6447  0.0     0.0       0.0
   4297.3      0   1 3.7847 3.7481 ...       0.0  14.4138  0.0     0.0       0.0
   4297.3      0   2 3.6626 1.7684 ...       0.0   13.753  0.0     0.0       0.0
   4297.3      0   3 3.7833 4.2391 ...       0.0  13.4045  0.0     0.0       0.0
   4297.3      0   4 3.6829 3.8664 ...       0.0  13.4159  0.0     0.0       0.0
   4297.3      0   5 3.6615 4.5936 ...       0.0  13.9712  0.0     0.0       0.0
   4297.3      0   6 3.7571 3.7104 ...       0.0  13.9862  0.0     0.0       0.0
   4297.3      0   7 3.6112  1.292 ...       0.0   13.207  0.0     0.0       0.0
   4297.3      0   8 3.7213 2.9467 ...       0.0  13.9755  0.0     0.0       0.0
   4297.3      0   9 3.7582 4.0112 ...       0.0  13.9673  0.0     0.0       0.0
      ...   

# PE table; create separate columns for each of the 15 emission lines

In [7]:
print('\nPE table:\n',tab_PE)
mask_elines = (tab_PE['model']=='eline')
tab_PE = tab_PE[mask_elines]
tab_PE.remove_columns(['id_fib','model'])
renames = {
           'flux'       : 'flux_pe',
           'e_flux'     : 'e_flux_pe',
           'disp'       : 'disp_pe',
           'e_disp'     : 'e_disp_pe',
           'vel'        : 'vel_pe',
           'e_vel'      : 'e_vel_pe'
}
for key in renames.keys():
    tab_PE.rename_column(key, renames[key])
print('\nPE renamed table columns:\n',tab_PE.colnames)
df0 = tab_PE.to_pandas()
df0.set_index(['id','wl'], inplace=True)
df1 = df0.unstack(level=-1)
df1.columns = [f"{c[0]}_{c[1]}" for c in df1.columns]
df2 = df1.reset_index()
tab_PE_ord = Table.from_pandas(df2)
print('\nPE_ord table:\n',tab_PE_ord)
print('\nPE_ord table columns:\n',tab_PE_ord.colnames)


PE table:
     id    id_fib model    wl      flux   ...  disp  e_disp    vel     e_vel 
--------- ------ ----- ------- --------- ... ------ ------ --------- -------
   4297.3      0 eline 6562.85   31.5686 ... 1.6865 0.0764  -24.0795  2.3907
   4297.3      0 eline 6583.45    1.3287 ... 1.6865 0.0764  -24.0795  2.3907
   4297.3      0 eline 6548.05    0.4424 ... 1.6865 0.0764  -24.0795  2.3907
   4297.3      0 eline 6730.82   12.1364 ... 0.3482 0.1367  -23.8256  6.1432
   4297.3      0 eline 6716.44   18.7195 ... 0.3482 0.1367  -23.8256  6.1432
   4297.3      0 eline 5006.84  339.9282 ... 1.6796 0.0917 -173.5826 22.3607
   4297.3      0 eline 4958.91  113.1961 ... 1.6796 0.0917 -173.5826 22.3607
   4297.3      0 eline 4861.36  339.9282 ... 1.6796 0.0917 -173.5826 22.3607
   4297.3      0 eline 4340.49   13.9455 ...   0.25    0.0   -84.957 21.6248
   4297.3      0 eline 4101.77   52.2639 ... 0.4919 0.0562 -135.4081  6.0744
      ...    ...   ...     ...       ... ...    ...    ...      

# NP tables (114 lines in B, 32 in R, 69 in I)

In [8]:
print('\nNPE_B table:\n',tab_NPE_B)
print('\nNPE_B table columns:\n',tab_NPE_B.colnames)
print('\nNPE_R table:\n',tab_NPE_R)
print('\nNPE_R table columns:\n',tab_NPE_R.colnames)
print('\nNPE_I table:\n',tab_NPE_I)
print('\nNPE_I table columns:\n',tab_NPE_I.colnames)


NPE_B table:
     id      flux_HI_3686.83   ... e_EW_[FeVII]_5720.7 e_EW_[NII]_5754.59
--------- ------------------- ... ------------------- ------------------
   4297.3  35.487430883143006 ...  2.6369365292860176  3.808995216101433
   4297.4    9.14697429044239 ...   6.499795591789285  8.110304983058036
   4297.5  44.023488573236854 ...   6.623853849192159  7.956918508379704
   4297.6  202.57023623301777 ...   7.630487084421976 22.209854555115015
   4297.7  221.76573618825978 ...  3.2182354200485745  4.911072446643607
   4297.8  168.68694786911016 ...  3.4345883570143445  6.984784845522497
   4297.9 -156.82185229609524 ...   3.964874954629629 5.3172049979089815
  4297.10  16.150809300651417 ...   3.469304554103542  4.198770264885706
  4297.11   128.4988647075219 ...    5.78346883838046  4.437694918375852
  4297.12 -22.985780644628157 ...  3.5434907951209658   4.82214482421027
      ...                 ... ...                 ...                ...
4297.1934   80.67487852907925 ...   

# KEL table (15 lines); create separate columns for each emission line

In [9]:
print('\nKEL table:\n',tab_KEL)
mask_elines = (tab_KEL['model']=='eline')
tab_KEL = tab_KEL[mask_elines]
tab_KEL.remove_columns(['id_fib','model'])
renames = {
           'flux'       : 'flux_pek',
           'e_flux'     : 'e_flux_pek',
           'disp'       : 'disp_pek',
           'e_disp'     : 'e_disp_pek',
           'vel'        : 'vel_pek',
           'e_vel'      : 'e_vel_pek'
}
for key in renames.keys():
    tab_KEL.rename_column(key, renames[key])
print('\nKEL renamed table columns:\n',tab_KEL.colnames)
df0 = tab_KEL.to_pandas()
df0.set_index(['id','wl'], inplace=True)
df1 = df0.unstack(level=-1)
df1.columns = [f"{c[0]}_{c[1]}" for c in df1.columns]
df2 = df1.reset_index()
tab_KEL_ord = Table.from_pandas(df2)
print('\nKEL_ord table:\n',tab_KEL_ord)
print('\nKEL_ord table columns:\n',tab_KEL_ord.colnames)


KEL table:
     id    id_fib model     wl     flux    e_flux  disp  e_disp   vel    e_vel 
--------- ------ ------ ------- -------- ------- ------ ------ -------- ------
   4297.3      0  eline 6562.85  43.7049 12.2156 0.3837 0.0141 -40.7713 2.0156
   4297.3      0  eline 6583.45   1.1205 11.2087 0.3837 0.0141 -40.7713 2.0156
   4297.3      0  eline 6548.05   0.3731 10.4261 0.3837 0.0141 -40.7713 2.0156
   4297.4      1  eline 6562.85   27.144 11.7596 0.3602 0.0189 -19.5523 2.3245
   4297.4      1  eline 6583.45  19.1236 12.1523 0.3602 0.0189 -19.5523 2.3245
   4297.4      1  eline 6548.05   6.3682 10.8542 0.3602 0.0189 -19.5523 2.3245
   4297.5      2  eline 6562.85   0.7202  9.9216 0.3765 0.0639 -73.4282 0.5972
   4297.5      2  eline 6583.45   0.7202 10.3242 0.3765 0.0639 -73.4282 0.5972
   4297.5      2  eline 6548.05   0.2398  9.7124 0.3765 0.0639 -73.4282 0.5972
   4297.6      3  eline 6562.85   0.7218 11.4412 0.3669 0.0075 -11.5425 3.0501
      ...    ...    ...     ...      ..

# SIGMA table (215 NP + 15 PE + 15 PEK)

In [10]:
print('\nSIGMA table:\n',tab_SIGMA)
print('\nSIGMA table columns:\n',tab_SIGMA.colnames)


SIGMA table:
     id    sigma_kms_HI_3686.83 ...  X_sq_pek_9069.0    X_sq_pek_9531.1  
--------- -------------------- ... ------------------ ------------------
  4297.10             87.08813 ... 2.6111163308880307  12.26815153927675
 4297.100            164.39073 ... 1.6139362814920697  9.082891845008323
4297.1000            39.608643 ... 52.563276615163446  105.0760707147916
4297.1001            37.794117 ...  58.35730855150671  93.89716231273182
4297.1002             39.26852 ... 41.682280246803145  92.67389793459874
4297.1003            37.620377 ... 57.643719825600456 107.43483466135233
4297.1004             38.92081 ...  60.17259001338994  152.3066370256854
4297.1005            39.515457 ...  92.15318500192983 153.33981850711245
4297.1006            38.943764 ...  78.71384646472129 202.41399086098681
4297.1007             39.53393 ...  97.70989202997282 221.37301909036523
      ...                  ... ...                ...                ...
 4297.990             35.14037 ... 2

# ELINES_CHI2_AVG table (245 rows)

In [11]:
print('\nELINES_CHI2_AVG table:\n', tab_CHIAVG)


ELINES_CHI2_AVG table:
      name        wave         inst_disp                X_sq       
-------------- ------- ------------------------ ------------------
    HI_3686.83 3686.83   0.7045149 .. 0.7519679  1.599146316780676
    HI_3691.56 3691.56 0.70394367 .. 0.75189155  1.651318101787818
    HI_3697.15 3697.15   0.7032442 .. 0.7517759  1.709927747055045
    HI_3703.85 3703.85  0.7023523 .. 0.75159377 1.6094139043928721
    HI_3711.97 3711.97  0.7013309 .. 0.75133854 29.114940110004884
 [OII]_3726.03 3726.03   0.6995393 .. 0.7507749  98.89562921677067
 [OII]_3728.82 3728.82  0.69915473 .. 0.7506352  99.00170072184092
    HI_3734.37 3734.37  0.6984496 .. 0.75036216   99.0825208438835
    HI_3750.15 3750.15    0.6964609 .. 0.749478 1.5866357875627852
[FeVII]_3758.9  3758.9  0.69530606 .. 0.7488893 1.6204536212923193
           ...     ...                      ...                ...
   pek_4958.91 4958.91   0.6646866 .. 0.6442791  52.34621186664779
   pek_5006.84 5006.84   0.6689838 ..

# INFO table

In [12]:
print('\nINFO table:\n')
tab_INFO.pprint(max_lines=-1,max_width=-1)


INFO table:

         param                                                 value                                       
------------------------ ----------------------------------------------------------------------------------
                lvm_file  /mnt/NAScoyote/LVM/spectro/redux/1.2.0/0011XX/11111/60191/lvmSFrame-00004297.fits
                   label                                                           dap-rsp108-sn20-00004297
             config_yaml                             _legacy/lvm-dap_SD_fast_SN20_ofelia_nosky_hack_cl.yaml
                   debug                                                                              False
                  lvmsim                                                                              False
                  in_rss                                                                              False
                    plot                                                                                  0
             o

# Merge the tables and output to CSV

In [13]:
tab_DAP=tab_PT.copy()
for t in [tab_RSP, tab_C_ord, tab_PE_ord, tab_KEL_ord, 
          tab_NPE_B, tab_NPE_R, tab_NPE_I, tab_SIGMA]:
    tab_DAP = join(tab_DAP, t, keys=['id'], join_type='left')
tab_DAP.sort(['exposure','fiberid'])
print('\nMerged DAP table:\n',tab_DAP)
print('\nMerged DAP table columns:\n',tab_DAP.colnames)
print(tab_DAP.meta)
tab_DAP.write('DAPtable_helix.ecsv', delimiter=',', overwrite=True)


Merged DAP table:
     id            ra         ...  X_sq_pek_9069.0    X_sq_pek_9531.1  
--------- ------------------ ... ------------------ ------------------
   4297.3  337.4169474614286 ... 1.3629693989829357 1.8276064927905626
   4297.4 337.42638153117906 ... 0.8842719614623011 2.1337073864328078
   4297.5 337.43581492731073 ... 1.8172620083840267 3.0613251971594515
   4297.6  337.4452142099591 ...  1.616723763279993 2.0232588773532894
   4297.7 337.45464625878174 ...  7.509199158175764 20.473380194511204
   4297.8 337.46407763267024 ...  1.172444005857491  1.689329535335471
   4297.9 337.47350833118605 ...   1.03862740097424  1.850846503869197
  4297.10 337.48293835389075 ... 2.6111163308880307  12.26815153927675
  4297.11 337.49233426716734 ... 0.9827429216924148 1.3101056605935648
  4297.12    337.50176293819 ... 3.3937517598670857  8.658537721161245
      ...                ... ...                ...                ...
4297.1934  337.2728561387537 ...  2.070920230862537  6.03